# Live Virtual Try-On in Colab/Kaggle (Gradio)

This notebook gives you a **live webcam UI** in browser using Gradio, similar to local testing.

- Uses your existing `RenderPipeline`
- Webcam input + live frame processing
- Shirt switch, opacity, shadows, lighting toggles


In [ ]:
import os, sys
from pathlib import Path

IS_COLAB = 'google.colab' in sys.modules
IS_KAGGLE = os.path.exists('/kaggle')
print('Colab:', IS_COLAB, '| Kaggle:', IS_KAGGLE)


In [ ]:
%%bash
set -e
python -m pip install --upgrade pip
python -m pip install gradio opencv-contrib-python ultralytics scipy colorlog matplotlib


In [ ]:
PROJECT_DIR = Path.cwd()
if not (PROJECT_DIR / 'engine').exists():
    for c in [Path('/content/Tryon'), Path('/kaggle/working/Tryon'), Path('/kaggle/input/Tryon')]:
        if (c / 'engine').exists():
            PROJECT_DIR = c
            break

assert (PROJECT_DIR / 'engine').exists(), f'engine/ not found in {PROJECT_DIR}'
os.chdir(PROJECT_DIR)
sys.path.insert(0, str(PROJECT_DIR))
print('Using PROJECT_DIR =', PROJECT_DIR)


In [ ]:
import cv2
import numpy as np
import threading
import gradio as gr

from engine.render_pipeline import RenderPipeline

pipeline = RenderPipeline(
    pose_model='yolov8n-pose.pt',
    device='auto',
    target_fps=30,
    enable_shadows=True,
    enable_lighting=True,
    opacity=0.95,
)

ok = pipeline.load_models()
print('load_models:', ok)
count = pipeline.load_garments(str(PROJECT_DIR / 'assets' / 'shirts'))
shirt_names = pipeline.get_shirt_names()
print('garments loaded:', count)
print('shirts:', shirt_names)

pipe_lock = threading.Lock()

def _shirt_name_to_index(name: str) -> int:
    try:
        return shirt_names.index(name)
    except Exception:
        return 0

def process_frame(frame_rgb, shirt_name, opacity, shadows, lighting):
    if frame_rgb is None:
        return None, 'No frame received'

    frame_bgr = cv2.cvtColor(frame_rgb, cv2.COLOR_RGB2BGR)

    with pipe_lock:
        pipeline.select_shirt(_shirt_name_to_index(shirt_name))
        pipeline.opacity = float(opacity)
        pipeline.enable_shadows = bool(shadows)
        pipeline.enable_lighting = bool(lighting)
        out_bgr, stats = pipeline.process_frame(frame_bgr)

    out_rgb = cv2.cvtColor(out_bgr, cv2.COLOR_BGR2RGB)
    status = (
        f'FPS: {stats.fps:.1f} | Pose: {stats.pose_detected} | '
        f'PoseMS: {stats.pose_ms:.1f} | WarpMS: {stats.warp_ms:.1f} | RenderMS: {stats.render_ms:.1f} | '
        f'Shirt: {stats.active_shirt}'
    )
    return out_rgb, status


In [ ]:
with gr.Blocks(title='Virtual Try-On Live (Colab)') as demo:
    gr.Markdown('## Virtual Try-On (Live Webcam)')

    with gr.Row():
        cam = gr.Image(label='Webcam Input', sources=['webcam'], type='numpy')
        out = gr.Image(label='Try-On Output', type='numpy')

    with gr.Row():
        shirt = gr.Dropdown(choices=shirt_names if shirt_names else ['shirt0'], value=(shirt_names[0] if shirt_names else 'shirt0'), label='Shirt')
        opac = gr.Slider(0.2, 1.0, value=0.95, step=0.01, label='Opacity')
        shadows = gr.Checkbox(value=True, label='Shadows')
        lighting = gr.Checkbox(value=True, label='Lighting')

    status = gr.Textbox(label='Status', interactive=False)

    # Live processing while webcam stream updates
    cam.change(
        fn=process_frame,
        inputs=[cam, shirt, opac, shadows, lighting],
        outputs=[out, status],
        queue=False,
    )

    # Manual fallback button (useful if live events are throttled)
    run_btn = gr.Button('Process Current Frame')
    run_btn.click(
        fn=process_frame,
        inputs=[cam, shirt, opac, shadows, lighting],
        outputs=[out, status],
        queue=False,
    )

share_flag = True if IS_COLAB else False
demo.launch(share=share_flag, debug=False, show_error=True)


In [ ]:
# Optional cleanup when finished
# pipeline.unload_models()
